In [1]:
import pandas as pd
import numpy as np
transactions = pd.read_csv('../data/processed/transactions.csv',
parse_dates=['invoice_date'])
customers = pd.read_csv('../data/processed/customers.csv')
sales = transactions[transactions['is_cancellation']==False].copy()
print('Sales rows:', len(sales))


Sales rows: 776629


In [2]:
snapshot = sales['invoice_date'].max() # 'today' for recency calculation
g = sales.groupby('customer_id')
features = pd.DataFrame({
'recency' : (snapshot - g['invoice_date'].max()).dt.days,
'frequency' : g['invoice'].nunique(),
'monetary' : g['line_revenue'].sum(),
'total_orders' : g['invoice'].nunique(),
'total_spend' : g['line_revenue'].sum(),
'unique_products': g['stock_code'].nunique(),
'total_items' : g['quantity'].sum(),
}).reset_index()
features['avg_order_value'] = features['total_spend'] / features['total_orders']
features['avg_basket_size'] = features['total_items'] / features['total_orders']
print(features.shape)
features.head()

(5861, 10)


,customer_id,recency,frequency,monetary,total_orders,total_spend,unique_products,total_items,avg_order_value,avg_basket_size
0,12346,325,12,77556.46,12,77556.46,27,74285,6463.038333,6190.416667
1,12347,1,8,4921.53,8,4921.53,126,2967,615.191250,370.875000
2,12348,74,5,1658.40,5,1658.40,24,2704,331.680000,540.800000
3,12349,18,3,3678.69,3,3678.69,137,1621,1226.230000,540.333333
4,12350,309,1,294.40,1,294.40,16,196,294.400000,196.000000


In [4]:
purchase_dates = (sales.groupby(['customer_id','invoice'])['invoice_date']
.min().reset_index()
.sort_values(['customer_id','invoice_date']))
purchase_dates['prev_date'] = purchase_dates.groupby('customer_id')['invoice_date'].shift(1)
purchase_dates['gap_days'] = (purchase_dates['invoice_date'] - purchase_dates['prev_date']).dt.days
avg_gap = (purchase_dates.groupby('customer_id')['gap_days']
.mean().rename('avg_days_between').reset_index())
features = features.merge(avg_gap, on='customer_id', how='left')
# One-time buyers have no gap — fill with a large number (they're risky)
features['avg_days_between'] = features['avg_days_between'].fillna(999)
print('avg_days_between added')

avg_days_between added


In [6]:

features['r_score'] = pd.qcut(features['recency'], 5, labels=[5,4,3,2,1]).astype(int)
features['f_score'] = pd.qcut(features['frequency'].rank(method='first'),
5, labels=[1,2,3,4,5]).astype(int)
features['m_score'] = pd.qcut(features['monetary'], 5, labels=[1,2,3,4,5]).astype(int)
def segment(row):
    r, f = row['r_score'], row['f_score']
    if r>=4 and f>=4: return 'Champion'
    if r>=3 and f>=3: return 'Loyal'
    if r>=4 and f<=2: return 'New/Promising'
    if r<=2 and f>=3: return 'At Risk'
    return 'Needs Attention'
features['segment'] = features.apply(segment, axis=1)
print(features['segment'].value_counts())

segment
Needs Attention    1894
Champion           1477
Loyal              1212
At Risk             827
New/Promising       451
Name: count, dtype: int64


In [7]:
features = features.merge(customers[['customer_id','country']], on='customer_id', how='left')

country_dummies = pd.get_dummies(features['country'], prefix='country', drop_first=True)
features = pd.concat([features, country_dummies], axis=1)
print('Final features shape:', features.shape)

Final features shape: (5861, 56)


In [8]:
features.to_csv('../data/processed/customer_features.csv', index=False)
print('Exported customer_features.csv')

Exported customer_features.csv
